# **Preprocesamiento para scikit-learn**

En esta sección se realiza el preprocesamiento de los datos para el flujo de modelado con scikit-learn.

Siguiendo los hallazgos del EDA, se seleccionan variables relevantes, se codifican las variables categóricas, se evalúa la necesidad de escalado en las variables numéricas y se realiza la partición del dataset en conjuntos de entrenamiento y prueba, utilizando una división estratificada 80/20 con respecto a la variable objetivo `default`.

In [11]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path(r"C:\Users\Daniel Rangel\Documents\MachineLearning\Data\accepted_2007_to_2018Q4.csv.gz")

df = pd.read_csv(DATA_PATH, low_memory=False)
df["default"] = df["loan_status"].apply(lambda x: 1 if x == "Charged Off" else 0)


## **Selección de variables**

De acuerdo con los hallazgos del análisis exploratorio, se selecciona un subconjunto de variables relevantes para la predicción de default.

Se trabajará con variables numéricas y categóricas que representan características del préstamo, perfil financiero del solicitante y contexto crediticio.

In [12]:
num_vars = [
    "loan_amnt",
    "int_rate",
    "fico_range_high",
    "annual_inc",
    "dti",
    "revol_util",
    "open_acc",
    "total_acc"
]

cat_vars = [
    "emp_length",
    "purpose",
    "home_ownership",
    "addr_state"
]

selected_vars = num_vars + cat_vars

df_model = df[selected_vars + ["default"]].copy()

print("Variables seleccionadas:")
print(selected_vars)
print("\nDimensión del subconjunto:", df_model.shape)

Variables seleccionadas:
['loan_amnt', 'int_rate', 'fico_range_high', 'annual_inc', 'dti', 'revol_util', 'open_acc', 'total_acc', 'emp_length', 'purpose', 'home_ownership', 'addr_state']

Dimensión del subconjunto: (2260701, 13)


## **Revisión inicial de valores faltantes**

Antes de construir el pipeline de preprocesamiento, se revisa la presencia de valores faltantes en las variables seleccionadas, con el fin de definir el tratamiento adecuado para variables numéricas y categóricas.

In [13]:
missing_model = df_model.isnull().sum()
missing_pct_model = (missing_model / len(df_model)) * 100

missing_model_df = pd.DataFrame({
    "missing": missing_model,
    "percentage": missing_pct_model
}).sort_values(by="percentage", ascending=False)

missing_model_df

,missing,percentage
emp_length,146940,6.499754
revol_util,1835,0.081170
dti,1744,0.077144
open_acc,62,0.002743
total_acc,62,0.002743
annual_inc,37,0.001637
loan_amnt,33,0.001460
int_rate,33,0.001460
fico_range_high,33,0.001460
purpose,33,0.001460


## **Tratamiento preliminar de valores faltantes**

Las variables seleccionadas presentan un nivel de valores faltantes relativamente bajo en general. La variable con mayor proporción de datos ausentes es `emp_length`, con aproximadamente 6.5%, mientras que en las variables numéricas restantes los porcentajes de missing son mínimos.

Dado este comportamiento, se considera adecuado aplicar estrategias simples de imputación dentro del pipeline de preprocesamiento:

- en variables numéricas, imputación mediante la mediana;
- en variables categóricas, imputación mediante la categoría más frecuente.

Este enfoque permite conservar el conjunto completo de observaciones y preparar los datos para el modelado con scikit-learn.

## **Construcción del pipeline de preprocesamiento**

A continuación se define un pipeline de preprocesamiento para scikit-learn.

Las variables numéricas serán imputadas y, posteriormente, escaladas con `StandardScaler`. Las variables categóricas serán imputadas y codificadas mediante `OneHotEncoder`, con el fin de transformarlas en una representación adecuada para los algoritmos de clasificación.

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [15]:
X = df_model[selected_vars]
y = df_model["default"]

print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

Shape de X: (2260701, 12)
Shape de y: (2260701,)


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train distribución:")
print(y_train.value_counts(normalize=True).round(4))
print("y_test distribución:")
print(y_test.value_counts(normalize=True).round(4))

X_train: (1808560, 12)
X_test: (452141, 12)
y_train distribución:
0    0.8812
1    0.1188
Name: default, dtype: float64
y_test distribución:
0    0.8812
1    0.1188
Name: default, dtype: float64


## **Partición de entrenamiento y prueba**

La división del dataset se realizó en una proporción 80/20, obteniendo **1,808,560** observaciones para entrenamiento y **452,141** para prueba.

La estratificación se aplicó correctamente, ya que tanto en `y_train` como en `y_test` se conserva la misma distribución de la variable objetivo: aproximadamente **88.12%** para la clase 0 y **11.88%** para la clase 1.

Esto es importante porque garantiza que ambos subconjuntos reflejen adecuadamente el desbalance presente en el dataset original, permitiendo una evaluación más consistente del desempeño del modelo.

In [17]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_vars),
        ("cat", categorical_transformer, cat_vars)
    ]
)

preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['loan_amnt', 'int_rate', 'fico_range_high',
                                  'annual_inc', 'dti', 'revol_util', 'open_acc',
                                  'total_acc']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['emp_length', 'purpose', 'home_ownership',
                                  'addr_state'])])

## **Definición del transformador de preprocesamiento**

El transformador de preprocesamiento combina el tratamiento de variables numéricas y categóricas en un único objeto `ColumnTransformer`.

De esta forma, las variables numéricas son imputadas con la mediana y escaladas con `StandardScaler`, mientras que las variables categóricas son imputadas con la categoría más frecuente y codificadas mediante `OneHotEncoder`.

In [18]:
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print("X_train_prepared:", X_train_prepared.shape)
print("X_test_prepared:", X_test_prepared.shape)

X_train_prepared: (1808560, 90)
X_test_prepared: (452141, 90)


## **Resultado del preprocesamiento**

Tras aplicar el pipeline de preprocesamiento, el conjunto de entrenamiento quedó representado por **90 variables transformadas**, mientras que el conjunto de prueba conserva la misma estructura.

El aumento en el número de variables se debe principalmente a la codificación one-hot de las variables categóricas, que transforma cada categoría en una representación binaria. De esta manera, los datos quedan listos para el entrenamiento de modelos en scikit-learn.